In [1]:
import pandas as pd
import numpy as np
import re

In [38]:
df = pd.read_csv('data/raw/INWE_2417_CTAB_20260424232823.csv', sep=';')

In [39]:
display(df.head())

,Kod,Nazwa,ogółem;2000;[zł],ogółem;2001;[zł],ogółem;2002;[zł],ogółem;2003;[zł],ogółem;2004;[zł],ogółem;2005;[zł],ogółem;2006;[zł],ogółem;2007;[zł],...,ogółem;2016;[zł],ogółem;2017;[zł],ogółem;2018;[zł],ogółem;2019;[zł],ogółem;2020;[zł],ogółem;2021;[zł],ogółem;2022;[zł],ogółem;2023;[zł],ogółem;2024;[zł],Unnamed: 27
0,200000,DOLNOŚLĄSKIE,3492,3786,3293,3129,3600,3970,5076,6044,...,7562,8358,9960,11590,10333,10468,11641,12212,11637,NaN
1,400000,KUJAWSKO-POMORSKIE,2318,2291,2312,2140,2251,2622,2981,3924,...,4338,4736,5851,5848,6361,7072,8456,9273,8909,NaN
2,600000,LUBELSKIE,1819,1684,1572,1637,1841,1992,2261,2803,...,3517,4145,5361,6040,6178,6287,6994,8839,8621,NaN
3,800000,LUBUSKIE,2604,2558,2455,2564,2727,3287,3537,4529,...,5573,5426,6207,6252,5992,8041,8830,12130,10402,NaN
4,1000000,ŁÓDZKIE,2501,2577,2274,2364,2798,3490,3889,5413,...,5744,5792,6723,7435,7026,7666,8701,10666,9139,NaN


In [40]:
df = df.iloc[:, 1:-1]

df.columns.values[0] = "region"

In [41]:
display(df.head())

,region,ogółem;2000;[zł],ogółem;2001;[zł],ogółem;2002;[zł],ogółem;2003;[zł],ogółem;2004;[zł],ogółem;2005;[zł],ogółem;2006;[zł],ogółem;2007;[zł],ogółem;2008;[zł],...,ogółem;2015;[zł],ogółem;2016;[zł],ogółem;2017;[zł],ogółem;2018;[zł],ogółem;2019;[zł],ogółem;2020;[zł],ogółem;2021;[zł],ogółem;2022;[zł],ogółem;2023;[zł],ogółem;2024;[zł]
0,DOLNOŚLĄSKIE,3492,3786,3293,3129,3600,3970,5076,6044,6420,...,7800,7562,8358,9960,11590,10333,10468,11641,12212,11637
1,KUJAWSKO-POMORSKIE,2318,2291,2312,2140,2251,2622,2981,3924,4817,...,6692,4338,4736,5851,5848,6361,7072,8456,9273,8909
2,LUBELSKIE,1819,1684,1572,1637,1841,1992,2261,2803,3526,...,4837,3517,4145,5361,6040,6178,6287,6994,8839,8621
3,LUBUSKIE,2604,2558,2455,2564,2727,3287,3537,4529,4382,...,5762,5573,5426,6207,6252,5992,8041,8830,12130,10402
4,ŁÓDZKIE,2501,2577,2274,2364,2798,3490,3889,5413,5792,...,6980,5744,5792,6723,7435,7026,7666,8701,10666,9139


In [42]:
unit_map = {}

new_cols = []
for col in df.columns:
    m = re.match(r"(.+?);(\d{4});\[(.+?)\]", col)
    
    if m:
        base = m.group(1)      # np. "ogółem"
        year = m.group(2)      # np. "2000"
        unit = m.group(3)      # np. "zł"

        new_name = f"{base}_{year}"
        new_cols.append(new_name)

        unit_map[new_name] = unit
    else:
        new_cols.append(col)

df.columns = new_cols

In [43]:
df_long = df.melt(
    id_vars=["region"], 
    var_name="zmienna", 
    value_name="inwestycje_zl"
)

df_long["jednostka"] = df_long["zmienna"].map(unit_map)

In [44]:
display(df_long)

,region,zmienna,inwestycje_zl,jednostka
0,DOLNOŚLĄSKIE,ogółem_2000,3492,zł
1,KUJAWSKO-POMORSKIE,ogółem_2000,2318,zł
2,LUBELSKIE,ogółem_2000,1819,zł
3,LUBUSKIE,ogółem_2000,2604,zł
4,ŁÓDZKIE,ogółem_2000,2501,zł
...,...,...,...,...
395,ŚLĄSKIE,ogółem_2024,10896,zł
396,ŚWIĘTOKRZYSKIE,ogółem_2024,7967,zł
397,WARMIŃSKO-MAZURSKIE,ogółem_2024,7741,zł
398,WIELKOPOLSKIE,ogółem_2024,10870,zł


In [45]:
df_long[["typ_inwestycji", "rok"]] = df_long["zmienna"].str.rsplit("_", n=1, expand=True)
df_long["rok"] = df_long["rok"].astype(int)

df_long.drop(columns=["zmienna", "jednostka"], inplace=True)

display(df_long)

,region,inwestycje_zl,typ_inwestycji,rok
0,DOLNOŚLĄSKIE,3492,ogółem,2000
1,KUJAWSKO-POMORSKIE,2318,ogółem,2000
2,LUBELSKIE,1819,ogółem,2000
3,LUBUSKIE,2604,ogółem,2000
4,ŁÓDZKIE,2501,ogółem,2000
...,...,...,...,...
395,ŚLĄSKIE,10896,ogółem,2024
396,ŚWIĘTOKRZYSKIE,7967,ogółem,2024
397,WARMIŃSKO-MAZURSKIE,7741,ogółem,2024
398,WIELKOPOLSKIE,10870,ogółem,2024


In [46]:
df_long.to_csv("data/processed/inwestycje.csv", index=False)

In [47]:
df_long.describe()

,inwestycje_zl,rok
count,400.000000,400.000000
mean,5793.050000,2012.000000
std,2919.966917,7.220133
min,1572.000000,2000.000000
25%,3597.500000,2006.000000
50%,5424.500000,2012.000000
75%,7280.500000,2018.000000
max,20911.000000,2024.000000


In [48]:
df_long.missing_values = df_long.isnull().sum()
display(df_long.missing_values)

C:\Users\user\AppData\Local\Temp\ipykernel_7468\278907725.py:1: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  df_long.missing_values = df_long.isnull().sum()


region            0
inwestycje_zl     0
typ_inwestycji    0
rok               0
dtype: int64